In [1]:
import torch
import open_clip
from PIL import Image
import chromadb
import os
from pathlib import Path
from dotenv import load_dotenv

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess, tokenizer = open_clip.create_model_and_transforms(
    "ViT-H-14",
    pretrained="laion2b_s32b_b79k"
)

model = model.to(device)
model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-31): 32 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1280, out_features=1280, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1280, out_features=5120, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=5120, out_features=1280, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1280,), eps=1e-05, elementwi

In [3]:
!nvidia-smi

Wed Feb 25 03:33:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.83                 Driver Version: 581.83         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650      WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   49C    P8              1W /   50W |    3873MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
def generate_image_embedding(image_path):
    image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)

    with torch.no_grad():
        features = model.encode_image(image)

    features = features / features.norm(dim=-1, keepdim=True)

    return features.cpu().numpy()[0]

In [5]:
load_dotenv()
CHROMA_PATH=os.getenv("CHROMA_LANDMARKS_DB_PATH")

path = Path(CHROMA_PATH)
client = chromadb.PersistentClient(path=path)
collection = client.create_collection(name="landmarks_images",get_or_create=True)

In [ ]:
import boto3

ACCOUNT_ID   = os.getenv("R2_ACCOUNT_ID")
ACCESS_KEY   = os.getenv("R2_ACCESS_KEY")
SECRET_KEY   = os.getenv("R2_SECRET_KEY")
BUCKET_NAME  = os.getenv("R2_BUCKET_NAME")

s3_client = boto3.client(
    "s3",
    region_name="auto",
    endpoint_url=f"https://{ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
)
PREFIX = "data/video_generation/raw/landmarks_images/"

objects = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=PREFIX)
image_files = [obj['Key'] for obj in objects.get('Contents', []) if obj['Key'].lower().endswith(('.jpg', '.png', '.jpeg'))]

image_id = 0
for key in image_files:
    obj = s3_client.get_object(Bucket=BUCKET_NAME, Key=key)
    image_bytes = obj['Body'] 

    emb = generate_image_embedding(image_bytes)
    
    landmark_name = key.split('/')[-2]  # parent folder
    file_name = key.split('/')[-1]

    collection.add(
        ids=[str(image_id)],
        embeddings=[emb],
        metadatas=[{
            "landmark": landmark_name,
            "path": str(key),
            "is_builder": "yes" if "builder" in file_name.lower() else "no",
            "is_plan": "yes" if "plan" in file_name.lower() else "no",
            "is_location": "yes" if "location" in file_name.lower() else "no"
        }],
    )
    image_id += 1

c:\Users\karim\AppData\Local\Programs\Python\Python39\lib\site-packages\boto3\compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


In [7]:
print(collection.count())

518


In [8]:
results = collection.get(limit=11)
for meta, doc in zip(results["metadatas"], results["documents"]):
    print(f"{doc} → {meta}")

None → {'is_plan': 'no', 'landmark': 'Al Qurn', 'path': 'data/video_generation/raw/landmarks_images/Al Qurn/1.jpg', 'is_builder': 'no', 'is_location': 'no'}
None → {'is_plan': 'no', 'landmark': 'Al Qurn', 'is_builder': 'no', 'is_location': 'no', 'path': 'data/video_generation/raw/landmarks_images/Al Qurn/10.jpg'}
None → {'is_builder': 'no', 'is_plan': 'no', 'is_location': 'no', 'path': 'data/video_generation/raw/landmarks_images/Al Qurn/3.jpg', 'landmark': 'Al Qurn'}
None → {'path': 'data/video_generation/raw/landmarks_images/Al Qurn/5.jpg', 'is_location': 'no', 'is_builder': 'no', 'is_plan': 'no', 'landmark': 'Al Qurn'}
None → {'is_plan': 'no', 'is_builder': 'no', 'is_location': 'no', 'landmark': 'Al Qurn', 'path': 'data/video_generation/raw/landmarks_images/Al Qurn/6.jpg'}
None → {'is_location': 'no', 'landmark': 'Al Qurn', 'is_plan': 'no', 'path': 'data/video_generation/raw/landmarks_images/Al Qurn/Al Qurn_20.jpg', 'is_builder': 'no'}
None → {'is_plan': 'no', 'path': 'data/video_gen

In [9]:
results = collection.get(include=["embeddings", "metadatas", "documents"], limit=2)
embeddings = results["embeddings"]
print(len(embeddings[0]))

1024
